# Pinned ArrowSpace TREC-COVID evaluation

This notebook builds pyarrowspace from an immutable source commit, verifies the published BEIR TREC-COVID archive, embeds documents and queries with a pinned MiniLM model, and evaluates cosine and ArrowSpace retrieval on all 50 test queries. It does not use the historical `Test-clean/` wrappers, whose targets are absent.

Qrels are used only for evaluation, not graph fitting or parameter selection. The tau sweep is predeclared; do not select and report the best test-set value as an independent result. This is descriptive benchmark evidence, not a generalisation claim, and it does not establish independence from MiniLM pretraining data.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import urllib.request

SOURCE_URL = "https://github.com/tuned-org-uk/pyarrowspace.git"
SOURCE_COMMIT = "e408abf46c727ede0415cec9b7d96274a71ad47a"
SOURCE_REF = "refs/tags/trec-covid-colab-runner"
RUST_TOOLCHAIN = "1.100.0"
SOURCE_ROOT = Path("/content/pyarrowspace-pinned")
ARTIFACT_ROOT = Path("/content/trec-covid-arrowspace-artifacts")

for path in (SOURCE_ROOT, ARTIFACT_ROOT):
    if path.exists():
        raise RuntimeError(f"Refusing to overwrite existing path: {path}; use a fresh Colab runtime")
print({"source_commit": SOURCE_COMMIT, "rust_toolchain": RUST_TOOLCHAIN})


## 1. Fetch and build the pinned implementation

The runner lives at the pinned commit, so the fetched tree contains both the package and the exact benchmark implementation. The notebook fetches a tag pointing to that commit, checks it out detached, and verifies the full commit SHA before building or running it. The SHA check fails closed if the tag is moved.


In [ ]:
def run(command, *, cwd=None, env=None):
    print("$", " ".join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)

run(["git", "init", str(SOURCE_ROOT)])
run(["git", "-C", SOURCE_ROOT, "remote", "add", "origin", SOURCE_URL])
run(["git", "-C", SOURCE_ROOT, "fetch", "--depth=1", "origin", SOURCE_REF])
run(["git", "-C", SOURCE_ROOT, "checkout", "--detach", "FETCH_HEAD"])
actual_source = subprocess.check_output(
    ["git", "-C", str(SOURCE_ROOT), "rev-parse", "HEAD"], text=True
).strip()
if actual_source != SOURCE_COMMIT:
    raise RuntimeError(f"Fetched {actual_source}; expected {SOURCE_COMMIT}")

if shutil.which("cargo") is None or shutil.which("rustup") is None:
    installer = Path("/content/rustup-init")
    urllib.request.urlretrieve(
        "https://static.rust-lang.org/rustup/dist/x86_64-unknown-linux-gnu/rustup-init",
        installer,
    )
    installer.chmod(0o700)
    run([installer, "-y", "--profile", "minimal", "--default-toolchain", RUST_TOOLCHAIN])
os.environ["PATH"] = str(Path.home() / ".cargo" / "bin") + os.pathsep + os.environ["PATH"]
os.environ["RUSTUP_TOOLCHAIN"] = RUST_TOOLCHAIN
run([sys.executable, "-m", "pip", "install", "uv"])
run(["uv", "sync", "--locked", "--extra", "embeddings"], cwd=SOURCE_ROOT)
PYTHON = SOURCE_ROOT / ".venv" / "bin" / "python"
run(["rustc", "--version"])
run([PYTHON, "-c", "import arrowspace; print('ArrowSpace import OK')"])


## 2. Validate, embed, and evaluate

The pinned runner checks the BEIR archive MD5 and task structure, hashes source inputs and generated embeddings into `manifest.json`, then computes cosine and ArrowSpace metrics without tuning on qrels. It writes no results until all validations and retrieval checks pass. The full run downloads about 170,000 documents and the pinned MiniLM model; use a fresh Colab runtime with sufficient memory and preferably a GPU.


In [ ]:
runner = SOURCE_ROOT / "neurips" / "Trec-covid" / "scripts" / "trec_covid_colab.py"
run([
    PYTHON,
    runner,
    "--output",
    ARTIFACT_ROOT,
    "--expected-source",
    SOURCE_COMMIT,
], cwd=SOURCE_ROOT)
results = json.loads((ARTIFACT_ROOT / "results.json").read_text())
assert results["real_trec_covid_results"] is True
assert results["source_commit"] == SOURCE_COMMIT
print(json.dumps(results["summary_macro"], indent=2, sort_keys=True))
print("Manifest:", ARTIFACT_ROOT / "manifest.json")
print("Task bundle:", ARTIFACT_ROOT / "trec-covid-task-bundle.zip")
print("Results:", ARTIFACT_ROOT / "results.json")


## Interpretation and artifacts

Download or copy `/content/trec-covid-arrowspace-artifacts/` to durable storage before the Colab runtime ends. Real results are marked as such only after the pinned runner finishes and verifies its outputs. `manifest.json` binds the exact source, dataset and model revisions, graph parameters, runtime versions, and output hashes. Results are descriptive TREC-COVID test-set outcomes; no claim is made about independent held-out performance or pretraining-data contamination.
